# Exploring Table Details and Relationships

**Notebook Version**: 1.0.0  
**Compatible with DBMCP**: 0.1.0+  
**Last Updated**: 2026-01-20  
**Test Database Version**: 1.0  
**Estimated Time**: 10 minutes  
**Difficulty**: Intermediate

## Overview

This notebook builds on basic connection skills to explore table schemas in depth. You'll learn how to inspect columns, data types, indexes, foreign keys, and understand table relationships.

## Prerequisites

- Completed [01_basic_connection.ipynb](01_basic_connection.ipynb) or familiar with database connections
- Understanding of database concepts: columns, primary keys, foreign keys
- Familiarity with data types (integers, strings, dates, etc.)

## What You'll Learn

- How to get detailed table schema information
- How to inspect column definitions, data types, and constraints
- How to examine indexes and their purposes
- How to discover declared foreign key relationships
- How to understand table relationships through constraints

## Environment Verification

In [ ]:
import sys
from pathlib import Path

print(f"Python version: {sys.version}")

# Find repository root and add to path
def find_repo_root():
    current = Path.cwd()
    for parent in [current] + list(current.parents):
        if (parent / "pyproject.toml").exists():
            return parent
    if current.name == "notebooks":
        return current.parent.parent
    return current

repo_root = find_repo_root()
sys.path.insert(0, str(repo_root))
print(f"Repository root: {repo_root}")

from examples.shared.notebook_helpers import verify_notebook_environment

verify_notebook_environment()

## Section 1: Connection Setup

Quick connection setup (refer to notebook 01 for details).

In [ ]:
from examples.shared.notebook_helpers import setup_connection

connection = setup_connection()
print("✓ Connection established")

## Section 2: Get Table Schema

Let's explore the detailed schema of a specific table. We'll examine the `customers` table from our test database.

**What we'll do**:
1. Get complete column information for the table
2. Display column names, data types, and constraints
3. Identify nullable vs required columns

In [ ]:
from sqlalchemy import inspect

from examples.shared.notebook_helpers import print_table

inspector = inspect(connection)
table_name = "customers"

# Get column information
columns = inspector.get_columns(table_name)

print(f"Schema for table '{table_name}':\n")

# Format column information
column_info = []
for col in columns:
    name = col['name']
    dtype = str(col['type'])
    nullable = "NULL" if col['nullable'] else "NOT NULL"
    default = col.get('default', 'None')
    column_info.append([name, dtype, nullable, default])

print_table(column_info, ["Column Name", "Data Type", "Nullable", "Default"])

**Understanding the results**:

This schema information shows:
- **Column Name**: The field name used to reference this data
- **Data Type**: What kind of data is stored (INTEGER for numbers, VARCHAR for text, etc.)
- **Nullable**: Whether the column can contain NULL (empty) values
- **Default**: Value automatically assigned if none is provided

**Key takeaways**:
- NOT NULL constraints indicate required fields
- Data types determine what operations are valid (can't add strings together as numbers)
- Default values help with data consistency

## Section 3: Inspect Indexes

Indexes speed up queries by allowing the database to find data faster. Let's see what indexes exist on our table.

**What we'll do**:
1. Get index information for the table
2. Identify unique vs non-unique indexes
3. Understand which columns are indexed

In [ ]:
# Get index information
indexes = inspector.get_indexes(table_name)

print(f"Indexes on '{table_name}' table:\n")

if indexes:
    index_info = []
    for idx in indexes:
        name = idx['name']
        columns = ", ".join(idx['column_names'])
        unique = "Yes" if idx['unique'] else "No"
        index_info.append([name, columns, unique])

    print_table(index_info, ["Index Name", "Columns", "Unique"])
else:
    print("  (No indexes defined beyond primary key)")

# Get primary key
pk = inspector.get_pk_constraint(table_name)
print(f"\nPrimary Key: {', '.join(pk['constrained_columns'])}")

**Understanding the results**:

Indexes are like a book's index - they help find information quickly:
- **Unique indexes** ensure no duplicate values (like email addresses)
- **Non-unique indexes** speed up searches without enforcing uniqueness
- **Primary keys** are automatically indexed and must be unique

**Key takeaways**:
- Indexes on foreign key columns speed up joins
- Too many indexes slow down INSERT/UPDATE operations
- Unique indexes enforce data integrity constraints

## Section 4: Inspect Foreign Keys

Foreign keys define relationships between tables. They ensure referential integrity - that related data actually exists.

**What we'll do**:
1. Examine foreign keys on the `orders` table
2. See which tables they reference
3. Understand the parent-child relationships

In [ ]:
# Check foreign keys on orders table
table_name = "orders"
foreign_keys = inspector.get_foreign_keys(table_name)

print(f"Foreign keys on '{table_name}' table:\n")

if foreign_keys:
    fk_info = []
    for fk in foreign_keys:
        name = fk.get('name', 'unnamed')
        local_cols = ", ".join(fk['constrained_columns'])
        ref_table = fk['referred_table']
        ref_cols = ", ".join(fk['referred_columns'])
        fk_info.append([local_cols, ref_table, ref_cols])

    print_table(fk_info, ["Local Column(s)", "References Table", "References Column(s)"])

    print("\nℹ️ This shows declared relationships - data integrity is enforced by the database")
else:
    print("  (No foreign keys declared)")

**Understanding the results**:

Foreign keys create relationships:
- **orders.customer_id → customers.customer_id** means each order belongs to a customer
- The database prevents orphaned records (orders with invalid customer_id)
- This is a **declared relationship** - explicitly defined in the schema

**Key takeaways**:
- Foreign keys enforce referential integrity
- They define one-to-many relationships (one customer, many orders)
- Understanding FKs is essential for writing correct JOIN queries

## Section 5: Discover Undeclared Relationships

Not all databases have foreign keys declared! Sometimes relationships exist through naming conventions and data patterns. Let's examine tables that might have undeclared relationships.

**What we'll do**:
1. Look at `shipping_addresses` table structure
2. Identify potential relationships by column names
3. Verify these are undeclared (no FK constraints)

In [ ]:
# Examine shipping_addresses table
table_name = "shipping_addresses"
columns = inspector.get_columns(table_name)

print(f"Columns in '{table_name}' table:\n")
for col in columns:
    print(f"  - {col['name']}: {col['type']}")

# Check for foreign keys
fks = inspector.get_foreign_keys(table_name)
print(f"\nDeclared foreign keys: {len(fks)}")

if len(fks) == 0:
    print("\n💡 Notice: This table has 'customer_id' but NO foreign key constraint!")
    print("This is an UNDECLARED relationship - the connection exists by naming convention.")
    print("\nWhy undeclared relationships exist:")
    print("  - Legacy databases without proper constraints")
    print("  - Performance considerations (FK checks add overhead)")
    print("  - Data warehouse designs (denormalized data)")
    print("  - Cross-database references (FK can't span databases)")

**Understanding the results**:

Undeclared relationships are common in real-world databases:
- Column name patterns suggest relationships (`customer_id`, `product_id`, etc.)
- Data types match between related columns
- But no FK constraint enforces the relationship

In the next notebook (03_advanced_patterns), we'll learn how DBMCP can automatically infer these hidden relationships using heuristics!

**Key takeaways**:
- Not all relationships are declared with foreign keys
- Naming conventions often indicate relationships (suffixes like `_id`)
- Relationship inference tools help discover undeclared relationships

## Summary

**What we covered**:
- ✓ Inspected detailed table schemas (columns, types, constraints)
- ✓ Examined indexes and their purposes
- ✓ Discovered declared foreign key relationships
- ✓ Identified undeclared relationships by naming patterns

**Key concepts**:
- **Schema Inspection**: Understanding table structure, columns, and data types
- **Indexes**: Performance optimization structures for faster queries
- **Foreign Keys**: Declared relationships that enforce referential integrity
- **Undeclared Relationships**: Hidden relationships indicated by naming patterns

**Common pitfalls**:
- ⚠️ **Missing FKs**: Not all databases have foreign keys declared - don't assume relationships are obvious
- ⚠️ **Index overhead**: While indexes speed up reads, they slow down writes
- ⚠️ **NULL handling**: Nullable columns require special care in queries (IS NULL vs = NULL)

## Next Steps

**Continue learning**:
- 📓 [03_advanced_patterns.ipynb](03_advanced_patterns.ipynb) - Relationship inference, error handling, and optimization
- 📓 [01_basic_connection.ipynb](01_basic_connection.ipynb) - Review basics if needed

**Explore further**:
- 📖 [SQLAlchemy Inspector Documentation](https://docs.sqlalchemy.org/en/14/core/reflection.html)
- 💡 Try inspecting different tables in the test database
- 💡 Compare the schema of tables with vs without foreign keys

**Need help?**
- 🐛 [Report issues](https://github.com/anthropics/dbmcp/issues)
- 💬 [Join discussions](https://github.com/anthropics/dbmcp/discussions)